# 🤖 Modeling & Prototyping
This notebook walks through the thought process of feature engineering, handling class imbalance, and selecting the best classification model (XGBoost).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', ''), errors='coerce').fillna(0)

# Drop ID
df.drop('customerID', axis=1, inplace=True)

# Target encoding
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## 2. Feature Engineering
Creating business-context features to help the model.

In [ ]:
def engineer_features(data):
    d = data.copy()
    d['TenureGroup'] = pd.cut(d['tenure'], bins=[-1, 12, 24, 48, 60, 100], labels=['0-1y', '1-2y', '2-4y', '4-5y', '5y+'])
    d['NumServices'] = (d[['PhoneService', 'InternetService', 'OnlineSecurity', 
                           'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                           'StreamingTV', 'StreamingMovies']] != 'No').sum(axis=1)
    d['MonthlyPerService'] = d['MonthlyCharges'] / (d['NumServices'] + 1)
    return d

df_eng = engineer_features(df)
display(df_eng.head(3))

## 3. Preprocessing Pipeline

In [ ]:
X = df_eng.drop('Churn', axis=1)
y = df_eng['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

## 4. Modeling (Baseline vs XGBoost)
Instead of using accuracy, we will focus on **Recall** (finding the churners) and **ROC-AUC**.

In [ ]:
# Random Forest Baseline
rf_pipe = Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# XGBoost Classifier
xgb_pipe = Pipeline([('prep', preprocessor), ('model', XGBClassifier(random_state=42, scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train)))])
xgb_pipe.fit(X_train, y_train)
y_pred_xgb = xgb_pipe.predict(X_test)
print('XGBoost Classification Report:')
print(classification_report(y_test, y_pred_xgb))

## 5. Decision Threshold Tuning
By default, `.predict()` uses a 0.5 threshold. For churn, we often want to lower this threshold to catch more potential churners (higher recall), even if it means some false positives.

In [ ]:
y_prob_xgb = xgb_pipe.predict_proba(X_test)[:, 1]
threshold = 0.4
y_pred_custom = (y_prob_xgb >= threshold).astype(int)

print(f'XGBoost with Custom Threshold ({threshold}):')
print(classification_report(y_test, y_pred_custom))

Lowering the threshold to 0.4 significantly improves our Recall for Class 1 (Churn), which translates into higher business ROI as demonstrated in the main application.